# 01. 병원 API 확인 — 응급의료기관 정보 조회

## 이 노트북이 하는 일
공공데이터포털 **응급의료정보 조회 서비스**(15000563)로 부산 동구 응급의료기관의 **등급·좌표**를 가져와, API 키가 동작하는지 확인한다. 이송 목적지(축3) 확정의 첫 단계.

## 왜 이렇게 설계했나 (설계 이유)
- **왜 이 API인가:** 심정지 이송 후보는 병원 '등급'으로 결정되는데, 이 API는 등급(`dutyEmclsName`)과 **좌표(wgs84)를 응답에 직접** 준다. 그래서 별도 지오코딩이 필요 없다.
- **왜 키를 .env에 두나:** 서비스키는 비밀값이라 코드에 하드코딩하면 유출된다. `.env`에 두고 `.gitignore`로 커밋 차단.
- **왜 Decoding 키인가:** `requests`가 `params`를 자동 URL 인코딩하므로, 이미 인코딩된 Encoding 키를 넣으면 이중 인코딩되어 인증 실패한다. → 반드시 Decoding 키.
- **왜 XML 파싱인가:** 이 서비스는 JSON을 안 주고 XML만 준다. 그래서 `xml.etree`로 파싱.
- **왜 Q0/Q1인가:** 목록·기본·위치 오퍼레이션의 지역 파라미터는 `Q0`(시도)·`Q1`(시군구)다. (`STAGE1/2`는 실시간 병상 전용 — 섞으면 필터가 무시돼 전국이 나온다.)

In [ ]:
# requests(HTTP 호출), python-dotenv(.env 로드), pandas(표 정리) 설치 — 최초 1회만
%pip install requests python-dotenv pandas

In [ ]:
import os                              # 환경변수(.env에서 로드한 키)에 접근하기 위해
import requests                         # 공공데이터포털 REST API를 GET으로 호출하기 위해
import xml.etree.ElementTree as ET      # 응답이 XML이라 파싱하기 위해 (이 API는 JSON 미지원)
import pandas as pd                     # 받은 기관 목록을 표(DataFrame)로 보기 좋게 정리하려고
from dotenv import load_dotenv          # 서비스키를 코드에 안 쓰고 .env 파일에서 읽어오려고

load_dotenv()                           # 같은 폴더의 .env를 읽어 환경변수로 등록 (키 유출 방지)

SERVICE_KEY = os.getenv("DATA_GO_KR_SERVICE_KEY")   # .env에 넣어둔 '일반 인증키(Decoding)'를 가져옴
assert SERVICE_KEY and SERVICE_KEY != "여기에_디코딩_키_붙여넣기", (   # 키가 비었거나 예시값이면 즉시 중단
    ".env 파일에 DATA_GO_KR_SERVICE_KEY 값을 실제 Decoding 키로 채워주세요."  # 사용자에게 원인 안내
)
print("키 로드 OK, 길이:", len(SERVICE_KEY))   # 키 자체는 출력 안 하고 길이만 찍어 로드 여부만 확인

BASE = "http://apis.data.go.kr/B552657/ErmctInfoInqireService"   # 응급의료기관 서비스의 공통 URL(엔드포인트 앞부분)

In [ ]:
def call_api(operation: str, **params):
    """오퍼레이션 이름과 파라미터를 받아 호출하고 (코드/메시지/항목/원문)을 돌려주는 공통 함수."""
    url = f"{BASE}/{operation}"                                  # 공통 URL 뒤에 오퍼레이션명을 붙여 최종 주소 완성
    q = {"serviceKey": SERVICE_KEY, "pageNo": 1, "numOfRows": 100, **params}  # 공통 파라미터(키·페이지·개수)에 개별 파라미터 병합
    r = requests.get(url, params=q, timeout=15)                  # GET 호출 (requests가 params를 자동 URL 인코딩 → Decoding 키 필요)
    r.raise_for_status()                                         # HTTP 오류(4xx/5xx)면 예외 발생시켜 조용한 실패 방지
    try:
        root = ET.fromstring(r.content)                         # 응답 XML 바이트를 파싱해 트리 루트 확보
    except ET.ParseError:                                        # 키 오류 등으로 XML이 아닌 응답이 올 때 대비
        return {"resultCode": None, "resultMsg": "XML 파싱 실패", "items": [], "raw": r.text}  # 원문을 담아 반환(디버깅용)
    header = root.find(".//header")                             # 응답 상태가 담긴 <header> 요소를 찾음
    code = header.findtext("resultCode") if header is not None else None   # 결과코드('00'=정상)
    msg = header.findtext("resultMsg") if header is not None else None     # 결과메시지('NORMAL SERVICE' 등)
    items = [{c.tag: c.text for c in item} for item in root.iter("item")]  # 모든 <item>을 {태그:값} dict 리스트로 변환
    return {"resultCode": code, "resultMsg": msg, "items": items, "raw": r.text}  # 4가지를 묶어 반환

## 테스트 1 — 응급의료기관 목록 (부산 동구)
`resultCode == '00'` 이면 키가 정상. 응답에 등급(`dutyEmclsName`)과 좌표(`wgs84Lat/Lon`)가 함께 들어 있다.

In [ ]:
# 목록/기본/위치 오퍼레이션은 Q0(시도)·Q1(시군구)를 쓴다 (STAGE1/2 아님 — 섞으면 전국이 나옴)
res = call_api("getEgytListInfoInqire", Q0="부산광역시", Q1="동구")   # 부산 동구 응급의료기관 목록 조회
print("resultCode:", res["resultCode"], "| resultMsg:", res["resultMsg"])  # '00' / 'NORMAL SERVICE'면 성공
print("기관 수:", len(res["items"]))                                  # 반환된 기관 개수

if res["resultCode"] != "00":                                        # 성공이 아니면
    print("\n--- 원문 응답 (앞 800자) ---\n", res["raw"][:800])     # 원문을 찍어 원인(키 에러 등) 확인

In [ ]:
# dutyEmclsName = 응급의료기관 분류(등급). 이 값으로 심정지 이송 후보인지 판단
df = pd.DataFrame(res["items"])                                       # item 리스트를 표로 변환
cols = [c for c in ["dutyName", "dutyEmclsName", "dutyAddr", "dutyTel1", "hpid"] if c in df.columns]  # 볼 컬럼만(있는 것만) 선택
df[cols] if not df.empty else "결과 없음 — resultCode 확인"          # 표 출력(비었으면 안내 문구)

## 테스트 2 — 좌표 확인
축3의 핵심. 좌표는 **위 목록 응답에 이미** `wgs84Lat/Lon`으로 들어 있어 별도 호출·지오코딩이 불필요하다. (위치정보 오퍼레이션 `getEgytLcinfoInqire`는 bounding box 방식이라 Q0/Q1로는 0건 → 여기선 안 씀.)

In [ ]:
# 좌표도 목록 응답(res)에 이미 포함되어 있으므로 그대로 뽑아서 표시
dfl = pd.DataFrame(res["items"])                                     # 같은 목록 데이터를 재사용
loc_cols = [c for c in ["dutyName", "dutyEmclsName", "wgs84Lon", "wgs84Lat", "dutyAddr"] if c in dfl.columns]  # 좌표 관련 컬럼 선택
dfl[loc_cols] if not dfl.empty else "결과 없음"                      # 병원명·등급·경위도·주소 표 출력

## 테스트 3 — 실시간 병상 (키 확인용)
신청 페이지에 안내됐던 `getEmrrmRltmUsefulSckbdInfoInqire`. **이 오퍼레이션만 STAGE1/STAGE2를 쓴다.** 본 프로젝트는 평시 도달시간 구조를 보므로 실제로는 안 쓰지만, 같은 키로 호출되는지 확인차 1회 호출.

In [ ]:
# 실시간 병상 오퍼레이션은 예외적으로 STAGE1/STAGE2 파라미터를 사용
rt = call_api("getEmrrmRltmUsefulSckbdInfoInqire", STAGE1="부산광역시", STAGE2="동구")  # 부산 동구 실시간 가용병상
print("resultCode:", rt["resultCode"], "| resultMsg:", rt["resultMsg"], "| 응답 item 수:", len(rt["items"]))  # 정상 여부·건수

## 테스트 4 — AED 취득 (축4, `aed_donggu.csv` 생성)

같은 제공기관(B552657)의 **AED 정보 조회 서비스**(15000652)로 부산 동구 AED를 받아 저장한다. 이 파일(`aed_donggu.csv`)은 축4 공백 분석(노트북 11)의 입력.
**주의:** AED는 병원과 **다른 데이터셋** → 별도 활용신청 필요. 키는 계정 공용(`.env`의 `AED_SERVICE_KEY`).

In [ ]:
import os                                                                              # (상단에서 이미 import했으나 명시)
AED_KEY = os.getenv("AED_SERVICE_KEY") or SERVICE_KEY                                 # AED 전용 키(없으면 병원 키 재사용)
AED_BASE = "http://apis.data.go.kr/B552657/AEDInfoInqireService"                      # AED 서비스 URL

def pull_aed(gu):                                                                     # 한 시군구의 AED 전량을 페이지 순회로 수집
    rows=[]; page=1
    while True:
        r=requests.get(f"{AED_BASE}/getEgytAedManageInfoInqire",                      # AED 관리기관 정보 조회
            params={"serviceKey":AED_KEY,"Q0":"부산광역시","Q1":gu,"pageNo":page,"numOfRows":100}, timeout=20)
        root=ET.fromstring(r.content)
        items=[{c.tag:c.text for c in it} for it in root.iter("item")]                # item → dict
        if not items: break
        rows+=items
        total=int(root.findtext(".//totalCount") or 0)                               # 총건수와 비교해 종료
        if len(rows)>=total: break
        page+=1
    return rows

aed = pd.DataFrame(pull_aed("동구"))                                                   # 부산 동구 AED 전량
aed["wgs84Lat"]=pd.to_numeric(aed["wgs84Lat"],errors="coerce")                       # 좌표 숫자화
aed["wgs84Lon"]=pd.to_numeric(aed["wgs84Lon"],errors="coerce")
os.makedirs("outputs", exist_ok=True)
aed.to_csv("outputs/aed_donggu.csv", index=False, encoding="utf-8-sig")               # 저장(노트북 11 입력)
print("부산 동구 AED:", len(aed), "개 | 좌표보유:", int(aed["wgs84Lat"].notna().sum()))
print("필드:", [c for c in ["org","buildPlace","buildAddress","wgs84Lon","wgs84Lat","monSttTme","monEndTme"] if c in aed.columns])

## 문제 해결

| 증상 | 원인 | 해결 |
|---|---|---|
| `SERVICE_KEY_IS_NOT_REGISTERED_ERROR` | Encoding 키를 넣음 / 발급 직후 | .env에 **Decoding** 키 / 몇 분 대기 |
| 전국 결과가 섞임 | 목록에 STAGE1/2 사용 | `Q0`/`Q1`로 변경 |
| AED 403 Forbidden | AED 데이터셋 미신청 | 15000652 별도 활용신청 |

정상이면 테스트 1의 `resultCode == '00'` + 테스트 4의 AED 저장이면 끝. 병원은 부산 구·군 순회로 이송 후보(권역·지역응급의료센터)를 뽑고, AED는 축4로 넘어간다.